# Import thư viện, Khởi tạo và Metric

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from sklearn.utils.class_weight import compute_class_weight

import gc
import cudf
import dask_cudf
import dask.array as da

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix
)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
class HybridDataset(Dataset):
    def __init__(self, X_seq, X_static, y):
        self.X_seq = torch.FloatTensor(X_seq)
        self.X_static = torch.FloatTensor(X_static)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_static[idx], self.y[idx]


class HybridGRUModel(nn.Module):
    def __init__(self, input_dim_per_phase, static_dim):
        super().__init__()

        # ===== GRU parameters =====
        self.hidden_dim = 128
        self.num_layers = 1
        self.dropout_p = 0.3
        self.num_classes = 3

        # ===== GRU layer =====
        self.gru = nn.GRU(
            input_size=input_dim_per_phase,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(self.dropout_p)

        # ===== Static branch=====
        self.mlp = nn.Sequential(
            nn.Linear(static_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # ===== Fully Connected layer =====
        # input_dim = hidden_dim (GRU last hidden state)
        self.fc = nn.Linear(self.hidden_dim + 32, self.num_classes)

    def forward(self, x_seq, x_static):
        # x_seq: (batch, seq_len, num_features)

        _, h_n = self.gru(x_seq)

        # h_n shape: (num_layers, batch, hidden_dim)
        # Lấy last hidden state
        h_last = h_n[-1]

        h_last = self.dropout(h_last)

        static_out = self.mlp(x_static)

        combined = torch.cat((h_last, static_out), dim=1)
        return self.fc(combined)


In [ ]:
# Metrics
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    per_class = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fn = cm[i].sum() - tp
        fp = cm[:,i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        per_class.append(np.sqrt(sens * spec))
    return np.prod(per_class) ** (1/len(per_class)) if per_class else 0

def gmean_per_class(y_true, y_pred, target_class):
    cm = confusion_matrix(y_true, y_pred)
    i = target_class
    tp = cm[i,i]
    fn = cm[i].sum() - tp
    fp = cm[:,i].sum() - tp
    tn = cm.sum() - tp - fn - fp

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return np.sqrt(recall * specificity)

def print_results(
    version_name,
    phase,
    y_true,
    y_pred,
    time_build_model=None,
    time_predict=None
):
    target_names = ['Excellent', 'Good', 'Average']

    print(f"\n{'='*30} {version_name} - Phase {phase} {'='*30}")
    print(classification_report(
        y_true,
        y_pred,
        digits=10,
        target_names=target_names
    ))

    # Precision / Recall / F1 theo từng class
    prec_per_class = precision_score(y_true, y_pred, average=None)
    rec_per_class  = recall_score(y_true, y_pred, average=None)
    f1_per_class   = f1_score(y_true, y_pred, average=None)

    # G-Mean per class
    gmean_classes = [
        gmean_per_class(y_true, y_pred, i)
        for i in range(len(target_names))
    ]

    print("G-Mean per class (one-vs-rest):")
    for idx, name in enumerate(target_names):
        print(f"  {name:<10}: {gmean_classes[idx]:.10f}")

    print()

    # ===== TẠO DICTIONARY METRICS =====
    metrics = {
        'Version': version_name,
        'Phase': phase,

        'TimeBuildModel': time_build_model,
        'TimePredict': time_predict,

        'Accuracy': accuracy_score(y_true, y_pred),
        'BalancedAcc': balanced_accuracy_score(y_true, y_pred),

        'Precision Macro': precision_score(y_true, y_pred, average='macro'),
        'Precision Weighted': precision_score(y_true, y_pred, average='weighted'),

        'Recall Macro': recall_score(y_true, y_pred, average='macro'),
        'Recall Weighted': recall_score(y_true, y_pred, average='weighted'),

        'F1-Score Macro': f1_score(y_true, y_pred, average='macro'),
        'F1-Score Weighted': f1_score(y_true, y_pred, average='weighted'),

        'GMean': gmean_score(y_true, y_pred),

        'MCC': matthews_corrcoef(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred),
    }

    # ===== THÊM METRIC CHO TỪNG CLASS =====
    for i, name in enumerate(target_names):
        metrics[f'Precision_{name}'] = prec_per_class[i]
        metrics[f'Recall_{name}'] = rec_per_class[i]
        metrics[f'F1-Score_{name}'] = f1_per_class[i]
        metrics[f'G-Mean_{name}'] = gmean_classes[i]

    # In ra console
    for k, v in metrics.items():
        if k not in ['Version', 'Phase'] and v is not None:
            print(f"{k:22} : {v:.10f}")

    return metrics

# Chuẩn bị dữ liệu + train

In [ ]:
# ===================== TRAIN (GRU VERSION - KEEP ORIGINAL) =====================
def prepare_and_train_hybrid(train_path, val_path, device, version_name):

    print(f"Loading train (GPU): {train_path}")
    ddf_train = dask_cudf.read_parquet(train_path)

    df_val = pd.read_parquet(val_path, engine='pyarrow') if val_path else None

    train_len = len(ddf_train)
    print(f"Train samples: {train_len}")
    if df_val is not None:
        print(f"Validation samples: {len(df_val)}")

    cols_to_drop = ['user_id', 'course_id']
    ddf_train = ddf_train.drop(columns=[c for c in cols_to_drop if c in ddf_train.columns])

    if df_val is not None:
        df_val = df_val.drop(columns=cols_to_drop, errors='ignore')

    y_train = ddf_train['label_3'].compute().to_numpy()
    X_train_ddf = ddf_train.drop('label_3', axis=1)

    if df_val is not None:
        y_val = df_val['label_3'].values
        X_val_df = df_val.drop('label_3', axis=1)

    train_columns = X_train_ddf.columns.tolist()
    phase_cols = [c for c in train_columns if any(f"_p{p}_" in c for p in ['1','2','3','4'])]
    static_cols = [c for c in train_columns if c not in phase_cols]

    for p in ['1','2','3','4']:
        print(f"Phase {p}: {len([c for c in phase_cols if f'_p{p}_' in c])} features")

    def build_seq_dask(ddf, p_cols):
        phases = []
        for p in ['1','2','3','4']:
            cols = sorted([c for c in p_cols if f"_p{p}_" in c])
            phases.append(ddf[cols].compute().to_numpy())
        return np.stack(phases, axis=1)

    def build_seq_pandas(df, p_cols):
        phases = []
        for p in ['1','2','3','4']:
            cols = sorted([c for c in p_cols if f"_p{p}_" in c])
            phases.append(df[cols].values)
        return np.stack(phases, axis=1)

    X_seq_train = build_seq_dask(X_train_ddf, phase_cols)
    X_static_train = X_train_ddf[static_cols].compute().to_numpy()

    print(f"Time-series shape: {X_seq_train.shape}")
    print(f"Static feature shape: {X_static_train.shape}")

    scaler_seq = StandardScaler()
    N, T, F = X_seq_train.shape
    X_seq_train = scaler_seq.fit_transform(X_seq_train.reshape(-1, F)).reshape(N, T, F)

    scaler_static = StandardScaler()
    X_static_train = scaler_static.fit_transform(X_static_train)

    if df_val is not None:
        X_seq_val = build_seq_pandas(X_val_df, phase_cols)
        X_static_val = X_val_df[static_cols].values
        N2 = X_seq_val.shape[0]
        X_seq_val = scaler_seq.transform(X_seq_val.reshape(-1, F)).reshape(N2, T, F)
        X_static_val = scaler_static.transform(X_static_val)

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    print(f"Classes: {le.classes_}")

    if df_val is not None:
        y_val_enc = le.transform(y_val)

    train_loader = DataLoader(HybridDataset(X_seq_train, X_static_train, y_train_enc), batch_size=256, shuffle=True)

    if df_val is not None:
        val_loader = DataLoader(HybridDataset(X_seq_val, X_static_val, y_val_enc), batch_size=256, shuffle=False)

    model = HybridGRUModel(F, X_static_train.shape[1]).to(device)

    unique_classes = np.unique(y_train_enc)
    weights = compute_class_weight('balanced', classes=unique_classes, y=y_train_enc)
    weights = torch.FloatTensor(weights).to(device)

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_loss = float('inf')
    patience = 10
    wait = 0
    start_train = time.perf_counter()

    for epoch in range(50):
        model.train()
        train_loss = 0
        for xb_seq, xb_static, yb in train_loader:
            xb_seq, xb_static, yb = xb_seq.to(device), xb_static.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb_seq, xb_static)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        if df_val is not None:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for xb_seq, xb_static, yb in val_loader:
                    xb_seq, xb_static, yb = xb_seq.to(device), xb_static.to(device), yb.to(device)
                    val_loss += criterion(model(xb_seq, xb_static), yb).item()
            val_loss /= len(val_loader)
            print(f"Epoch {epoch+1}: train = {train_loss:.4f}, val = {val_loss:.4f}")
            monitor = val_loss
        else:
            print(f"Epoch {epoch+1}: loss = {train_loss:.4f}")
            monitor = train_loss

        if monitor < best_loss - 1e-4:
            best_loss = monitor
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience: break

    model.load_state_dict(best_state)
    time_build = time.perf_counter() - start_train

    os.makedirs("saved_models", exist_ok=True)
    torch.save(model.state_dict(), f"saved_models/GRU_{version_name}.pt")

    return model, scaler_seq, scaler_static, le, phase_cols, static_cols, time_build

# Chạy theo từng V

In [ ]:
def run_experiment(base_path, train_file, val_file, test_prefix, version_name, device = None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"\n{'#'*20}")
    print(f"Version: {version_name}")
    print(f"{'#'*20}")

    model, scaler_seq, scaler_static, le, phase_cols, static_cols, time_build = \
        prepare_and_train_hybrid(f"{base_path_1}/{train_file}", f"{base_path}/{val_file}", device, version_name)

    results = []
    model.eval()

    for phase in range(1, 5):
        test_path = f"{base_path}/{test_prefix}_{phase}.parquet"
        print(f"\n--- Test Phase {phase} (Parquet): {test_path} ---")

        df = pd.read_parquet(test_path, engine='pyarrow')

        df = df.drop(columns=['user_id','course_id'], errors='ignore')
        y_raw = df['label_3'].values
        X_df = df.drop('label_3', axis=1)

        del df
        gc.collect()

        def build_seq_local(df_input):
            phases_list = []
            for p in ['1','2','3','4']:
                cols = sorted([c for c in phase_cols if f"_p{p}_" in c])
                phases_list.append(df_input[cols].values)
            return np.stack(phases_list, axis=1)

        X_seq = build_seq_local(X_df)
        X_static = X_df[static_cols].values

        del X_df
        gc.collect()

        N, T, F = X_seq.shape
        X_seq = scaler_seq.transform(X_seq.reshape(-1, F)).reshape(N, T, F)
        X_static = scaler_static.transform(X_static)

        y_enc = le.transform(y_raw)

        test_dataset = HybridDataset(X_seq, X_static, y_enc)
        test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

        all_probs = []
        start_pred = time.perf_counter()

        with torch.no_grad():
            for xb_seq, xb_static, _ in test_loader:
                xb_seq = xb_seq.to(device)
                xb_static = xb_static.to(device)

                outputs = model(xb_seq, xb_static)

                prob = torch.softmax(outputs, dim=1).cpu().numpy()
                all_probs.append(prob)

                del xb_seq, xb_static, outputs

        time_pred = time.perf_counter() - start_pred

        probs = np.vstack(all_probs)
        preds = np.argmax(probs, axis=1)

        metrics = print_results(version_name, phase, y_enc, preds, time_build, time_pred)

        os.makedirs("results_GRU", exist_ok=True)
        cm = confusion_matrix(y_enc, preds)
        pd.DataFrame(cm).to_csv(f"results_GRU/confusion_matrix_{version_name}_phase{phase}.csv", index=False)

        df_prob = pd.DataFrame(probs, columns=[f"Prob_Class_{i}" for i in range(probs.shape[1])])
        df_prob['y_true'] = y_enc
        df_prob['y_pred'] = preds
        df_prob.to_csv(f"results_GRU/probability_matrix_{version_name}_phase{phase}.csv", index=False)

        results.append(metrics)

        del X_seq, X_static, y_raw, y_enc, all_probs, probs, preds, df_prob
        torch.cuda.empty_cache()
        gc.collect()

    df_res = pd.DataFrame(results).round(10)
    ordered_cols = [
        "Version","Phase","TimeBuildModel","TimePredict","Accuracy","BalancedAcc",
        "Precision Macro","Precision Weighted","Recall Macro","Recall Weighted",
        "F1-Score Macro","F1-Score Weighted","GMean","MCC","Kappa",
        "Precision_Excellent","Recall_Excellent","F1-Score_Excellent","G-Mean_Excellent",
        "Precision_Good","Recall_Good","F1-Score_Good","G-Mean_Good",
        "Precision_Average","Recall_Average","F1-Score_Average","G-Mean_Average"
    ]

    final_cols = [c for c in ordered_cols if c in df_res.columns]
    df_res = df_res[final_cols]

    return df_res

## V_Median

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v1 = run_experiment(
    base_path=base_path,
    train_file="train_median.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V1 (Median)"
)

df_v1


####################
Version: V1 (Median)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/train_median.parquet
Train samples: 1859619
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (1859619, 4, 39)
Static feature shape: (1859619, 23)
Classes: [0 1 2]
Epoch 1: train = 0.2494, val = 0.1453
Epoch 2: train = 0.1660, val = 0.1156
Epoch 3: train = 0.1466, val = 0.1159
Epoch 4: train = 0.1316, val = 0.1208
Epoch 5: train = 0.1212, val = 0.1177
Epoch 6: train = 0.1179, val = 0.1106
Epoch 7: train = 0.1094, val = 0.1079
Epoch 8: train = 0.1074, val = 0.1175
Epoch 9: train = 0.1039, val = 0.1154
Epoch 10: train = 0.0958, val = 0.1144
Epoch 11: train = 0.0962, val = 0.1142
Epoch 12: train = 0.0935, val = 0.1276
Epoch 13: train = 0.0895, val = 0.1258
Epoch 14: train = 0.0893, val = 0.1180
Epoch 15: train = 0.0881, val = 0.1175
Epoch 16: train = 0.0862, val = 0.131

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V1 (Median),1,712.910375,3.416277,0.842863,0.829786,0.354507,0.996252,0.829786,0.842863,0.345479,0.911614,0.875401,0.118581,0.032180,0.049732,0.874439,0.094112,0.927583,0.014060,0.771901,0.027617,0.814169,0.999729,0.843018,0.914710,0.888289
1,V1 (Median),2,712.910375,3.407291,0.790263,0.852557,0.358327,0.996365,0.852557,0.790263,0.341413,0.879633,0.881980,0.106596,0.024382,0.063990,0.914798,0.119613,0.950286,0.011155,0.852893,0.022023,0.827424,0.999836,0.789979,0.882605,0.872558
2,V1 (Median),3,712.910375,3.399876,0.849789,0.915118,0.355467,0.996496,0.915118,0.849789,0.348722,0.915504,0.929392,0.137342,0.038161,0.048474,0.968610,0.092327,0.975154,0.017963,0.927273,0.035243,0.897000,0.999964,0.849472,0.918595,0.917764
3,V1 (Median),4,712.910375,3.452251,0.990712,0.962332,0.561039,0.997398,0.962332,0.990712,0.660101,0.993389,0.975783,0.510095,0.422069,0.450106,0.950673,0.610951,0.974481,0.233089,0.945455,0.373978,0.968390,0.999922,0.990869,0.995375,0.984545


In [ ]:
df_v1.to_csv("results_v1.csv", index=False)

## V_SMOTE

In [ ]:
base_path_1 = "/kaggle/input/datasets/uyentran10/lo-smote-test"

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v22 = run_experiment(
    base_path=base_path,
    train_file="train_median_smote.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V22 (Median SMOTE)"
)
df_v22


####################
Version: V22 (Median SMOTE)
####################
Loading train (GPU): /kaggle/input/datasets/uyentran10/lo-smote-test/train_median_smote.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0527, val = 0.0380
Epoch 2: train = 0.0267, val = 0.0333
Epoch 3: train = 0.0226, val = 0.0296
Epoch 4: train = 0.0205, val = 0.0298
Epoch 5: train = 0.0191, val = 0.0284
Epoch 6: train = 0.0181, val = 0.0281
Epoch 7: train = 0.0173, val = 0.0285
Epoch 8: train = 0.0167, val = 0.0265
Epoch 9: train = 0.0161, val = 0.0290
Epoch 10: train = 0.0157, val = 0.0275
Epoch 11: train = 0.0154, val = 0.0281
Epoch 12: train = 0.0150, val = 0.0276
Epoch 13: train = 0.0148, val = 0.0265
Epoch 14: train = 0.0145, val = 0.0256
Epoch 15: train = 0.0143, val = 0.0241
Epoch 16: train = 0.0141, val = 0.

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V22 (Median SMOTE),1,3335.052971,3.220076,0.994291,0.582225,0.495037,0.995041,0.582225,0.994291,0.517887,0.994604,0.589601,0.292191,0.290151,0.220114,0.520179,0.309333,0.720596,0.267308,0.229752,0.247111,0.478931,0.997688,0.996745,0.997216,0.593896
1,V22 (Median SMOTE),2,3335.052971,3.227337,0.995539,0.675646,0.586804,0.996120,0.675646,0.995539,0.604782,0.995742,0.700526,0.427359,0.425733,0.287671,0.659193,0.400545,0.811270,0.474576,0.370248,0.415970,0.608154,0.998164,0.997496,0.997830,0.696776
2,V22 (Median SMOTE),3,3335.052971,3.218122,0.991598,0.818914,0.492532,0.996447,0.818914,0.991598,0.568786,0.993642,0.854341,0.427920,0.376481,0.198907,0.816143,0.319859,0.901980,0.279601,0.647934,0.390633,0.803188,0.999087,0.992665,0.995866,0.860757
3,V22 (Median SMOTE),4,3335.052971,3.228858,0.994468,0.921728,0.584420,0.997480,0.921728,0.994468,0.684805,0.995607,0.945223,0.584292,0.534374,0.385496,0.905830,0.540830,0.951091,0.368051,0.864463,0.516288,0.927963,0.999714,0.994893,0.997297,0.956864


In [ ]:
df_v22.to_csv("results_v22.csv", index=False)

## V_GAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/anhtran10/lo-gan-test"

In [ ]:
df_v23 = run_experiment(
    base_path=base_path,
    train_file="train_median_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V23 (Median GAN)"
)
df_v23


####################
Version: V23 (Median GAN)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-gan-test/train_median_gan.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0035, val = 0.0071
Epoch 2: train = 0.0022, val = 0.0053
Epoch 3: train = 0.0020, val = 0.0053
Epoch 4: train = 0.0019, val = 0.0052
Epoch 5: train = 0.0018, val = 0.0051
Epoch 6: train = 0.0017, val = 0.0050
Epoch 7: train = 0.0017, val = 0.0049
Epoch 8: train = 0.0016, val = 0.0047
Epoch 9: train = 0.0016, val = 0.0048
Epoch 10: train = 0.0016, val = 0.0051
Epoch 11: train = 0.0016, val = 0.0047
Epoch 12: train = 0.0015, val = 0.0047
Epoch 13: train = 0.0015, val = 0.0050
Epoch 14: train = 0.0015, val = 0.0047
Epoch 15: train = 0.0015, val = 0.0048
Epoch 16: train = 0.0015, val = 0.0052
Ep

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V23 (Median GAN),1,2019.33411,3.164565,0.992906,0.550449,0.492313,0.994920,0.550449,0.992906,0.513553,0.993851,0.583342,0.263544,0.254795,0.315574,0.345291,0.329764,0.587404,0.163621,0.310744,0.214367,0.556287,0.997745,0.995311,0.996527,0.607482
1,V23 (Median GAN),2,2019.33411,3.229635,0.995793,0.629987,0.601484,0.995869,0.629987,0.995793,0.613988,0.995826,0.664172,0.419024,0.418947,0.421818,0.520179,0.465863,0.720988,0.384615,0.371901,0.378151,0.609363,0.998018,0.997880,0.997949,0.666864
2,V23 (Median GAN),3,2019.33411,3.191678,0.994584,0.738068,0.642515,0.996374,0.738068,0.994584,0.676994,0.995375,0.776371,0.449192,0.433835,0.662447,0.704036,0.682609,0.838924,0.266495,0.514050,0.351016,0.715648,0.998602,0.996119,0.997359,0.779446
3,V23 (Median GAN),4,2019.33411,3.236336,0.998451,0.823967,0.874790,0.998338,0.823967,0.998451,0.846597,0.998373,0.854654,0.767486,0.765166,0.847534,0.847534,0.847534,0.920549,0.777778,0.624793,0.692942,0.790255,0.999059,0.999573,0.999316,0.858138


In [ ]:
df_v23.to_csv("results_v23.csv", index=False)

## V_CDSMOTE

In [ ]:
df_v2 = run_experiment(
    base_path=base_path,
    train_file="train_median_cdsmote.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V2 (Median CDSMOTE)"
)

df_v2


####################
Version: V2 (Median CDSMOTE)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/train_median_cdsmote.parquet
Train samples: 5558987
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558987, 4, 39)
Static feature shape: (5558987, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0380, val = 0.0241
Epoch 2: train = 0.0197, val = 0.0195
Epoch 3: train = 0.0167, val = 0.0237
Epoch 4: train = 0.0151, val = 0.0195
Epoch 5: train = 0.0141, val = 0.0251
Epoch 6: train = 0.0134, val = 0.0212
Epoch 7: train = 0.0128, val = 0.0254
Epoch 8: train = 0.0123, val = 0.0216
Epoch 9: train = 0.0119, val = 0.0216
Epoch 10: train = 0.0116, val = 0.0211
Epoch 11: train = 0.0114, val = 0.0187
Epoch 12: train = 0.0110, val = 0.0208
Epoch 13: train = 0.0110, val = 0.0191
Epoch 14: train = 0.0108, val = 0.0196
Epoch 15: train = 0.0107, val = 0.0194
Epoch 16: train = 0.0

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V2 (Median CDSMOTE),1,3233.35772,3.426257,0.995801,0.498508,0.562712,0.994814,0.498508,0.995801,0.521140,0.995235,0.472252,0.268980,0.260206,0.386139,0.349776,0.367059,0.591261,0.304795,0.147107,0.198439,0.383378,0.997202,0.998640,0.997921,0.464639
1,V2 (Median CDSMOTE),2,3233.35772,3.427752,0.995780,0.642587,0.602198,0.995698,0.642587,0.995780,0.615439,0.995713,0.650540,0.399134,0.399095,0.440625,0.632287,0.519337,0.794858,0.368098,0.297521,0.329068,0.545091,0.997872,0.997954,0.997913,0.635422
2,V2 (Median CDSMOTE),3,3233.35772,3.393052,0.993551,0.811970,0.531175,0.996465,0.811970,0.993551,0.608161,0.994762,0.837713,0.458367,0.425915,0.270804,0.860987,0.412017,0.926860,0.323801,0.580165,0.415631,0.760481,0.998920,0.994759,0.996835,0.834033
3,V2 (Median CDSMOTE),4,3233.35772,3.399115,0.995169,0.908535,0.597634,0.997522,0.908535,0.995169,0.694608,0.996058,0.934580,0.601435,0.561690,0.378788,0.896861,0.532623,0.946358,0.414474,0.833058,0.553542,0.911317,0.999640,0.995687,0.997660,0.946508


In [ ]:
df_v2.to_csv("results_v2.csv", index=False)

## V_SMOTified GAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/uyentran10/lo-smotifiedgan-test"

In [ ]:
df_v24 = run_experiment(
    base_path=base_path,
    train_file="train_median_smotified_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V24 (Median SMOTified GAN)"
)
df_v24


####################
Version: V24 (Median SMOTified GAN)
####################
Loading train (GPU): /kaggle/input/datasets/uyentran10/lo-smotifiedgan-test/train_median_smotified_gan.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0037, val = 0.0059
Epoch 2: train = 0.0022, val = 0.0051
Epoch 3: train = 0.0020, val = 0.0052
Epoch 4: train = 0.0019, val = 0.0049
Epoch 5: train = 0.0018, val = 0.0049
Epoch 6: train = 0.0017, val = 0.0050
Epoch 7: train = 0.0017, val = 0.0050
Epoch 8: train = 0.0016, val = 0.0048
Epoch 9: train = 0.0016, val = 0.0048
Epoch 10: train = 0.0016, val = 0.0049
Epoch 11: train = 0.0016, val = 0.0049
Epoch 12: train = 0.0015, val = 0.0049
Epoch 13: train = 0.0015, val = 0.0049
Epoch 14: train = 0.0015, val = 0.0049

--- Test Phase 1 (Parquet): /kaggle/input/dataset

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V24 (Median SMOTified GAN),1,1588.831405,3.219497,0.995010,0.527535,0.510649,0.994937,0.527535,0.995010,0.508805,0.994926,0.531839,0.277758,0.277620,0.221053,0.376682,0.278607,0.613353,0.313433,0.208264,0.250248,0.456088,0.997462,0.997660,0.997561,0.537752
1,V24 (Median SMOTified GAN),2,1588.831405,3.235092,0.996507,0.625531,0.659313,0.996344,0.625531,0.996507,0.619963,0.996283,0.650304,0.454011,0.450805,0.366569,0.560538,0.443262,0.748343,0.613419,0.317355,0.418301,0.563196,0.997951,0.998700,0.998326,0.652514
2,V24 (Median SMOTified GAN),3,1588.831405,3.298277,0.997522,0.758309,0.749779,0.997503,0.758309,0.997522,0.741285,0.997447,0.791597,0.634840,0.634031,0.534375,0.766816,0.629834,0.875399,0.716279,0.509091,0.595169,0.713318,0.998684,0.999020,0.998852,0.794370
3,V24 (Median SMOTified GAN),4,1588.831405,3.222925,0.998456,0.831447,0.888221,0.998332,0.831447,0.998456,0.854042,0.998349,0.850631,0.764852,0.761018,0.855319,0.901345,0.877729,0.949323,0.810384,0.593388,0.685115,0.770178,0.998960,0.999607,0.999284,0.841819


In [ ]:
df_v24.to_csv("results_v24.csv", index=False)

## V_CDSGAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/hngliththu/cdsmote-gan"

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v13 = run_experiment(
    base_path=base_path,
    train_file="train_median_cdsmote_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V13 (Median CDSMOTified GAN)"
)
df_v13


####################
Version: V13 (Median CDSMOTified GAN)
####################
Loading train (GPU): /kaggle/input/datasets/hngliththu/cdsmote-gan/train_median_cdsmote_gan.parquet
Train samples: 5558987
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558987, 4, 39)
Static feature shape: (5558987, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0036, val = 0.0058
Epoch 2: train = 0.0021, val = 0.0051
Epoch 3: train = 0.0019, val = 0.0047
Epoch 4: train = 0.0018, val = 0.0047
Epoch 5: train = 0.0017, val = 0.0048
Epoch 6: train = 0.0017, val = 0.0049
Epoch 7: train = 0.0016, val = 0.0047
Epoch 8: train = 0.0016, val = 0.0048
Epoch 9: train = 0.0015, val = 0.0047
Epoch 10: train = 0.0015, val = 0.0045
Epoch 11: train = 0.0015, val = 0.0047
Epoch 12: train = 0.0015, val = 0.0047
Epoch 13: train = 0.0015, val = 0.0047
Epoch 14: train = 0.0014, val = 0.0046
Epoch 15: train = 0.0014, val = 0.0049
Epoch 16: train = 0.

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V13 (Median CDSMOTified GAN),1,2285.386438,3.153888,0.996008,0.480943,0.695921,0.995297,0.480943,0.996008,0.535430,0.995484,0.476047,0.299344,0.288428,0.769231,0.224215,0.347222,0.473498,0.321256,0.219835,0.261040,0.468581,0.997276,0.998778,0.998026,0.486236
1,V13 (Median CDSMOTified GAN),2,2285.386438,3.173228,0.996989,0.537455,0.814939,0.996296,0.537455,0.996989,0.619839,0.996373,0.549250,0.444471,0.407264,0.870968,0.363229,0.512658,0.602669,0.576336,0.249587,0.348328,0.499467,0.997514,0.999551,0.998531,0.550458
2,V13 (Median CDSMOTified GAN),3,2285.386438,3.202369,0.997320,0.665645,0.811375,0.996827,0.665645,0.997320,0.723188,0.996973,0.681241,0.551262,0.536488,0.850575,0.663677,0.745592,0.814618,0.585507,0.333884,0.425263,0.577649,0.998043,0.999374,0.998708,0.671869
3,V13 (Median CDSMOTified GAN),4,2285.386438,3.142121,0.998438,0.803869,0.903659,0.998296,0.803869,0.998438,0.844064,0.998291,0.826942,0.756520,0.748536,0.872146,0.856502,0.864253,0.925418,0.840000,0.555372,0.668657,0.745130,0.998831,0.999732,0.999281,0.820076


In [ ]:
df_v13.to_csv("results_v13.csv", index=False)